In [18]:
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import os
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

/Users/zrutix/Projects/Bakalarka/gat-document-relation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_node_features(graph):
    image_W = graph['width']
    image_H = graph['height']

    filtered_nodes = []
    features = []

    for node in graph['nodes']:
        x, y, w, h = node['bbox']
        area = (w * h) / (image_W * image_H)

        # remove tiny text boxes
        if node['category_id'] == 10 and area < 0.001:
            continue

        filtered_nodes.append(node)

        x_center = (x + w/2) / image_W
        y_center = (y + h/2) / image_H
        w_norm = w / image_W
        h_norm = h / image_H
        aspect = w / (h + 1e-6)

        category_id = np.zeros(11)
        category_id[node['category_id'] - 1] = 1

        geom_features = np.array([
            x_center,
            y_center,
            w_norm,
            h_norm,
            area,
            aspect
        ])

        feature = np.concatenate([geom_features, category_id])
        features.append(feature)

    return np.array(features), filtered_nodes

In [4]:
def build_edge_index(edges, filtered_nodes):
    id_map = {}
    # 1. Build map ONLY from the filtered nodes
    for idx, node in enumerate(filtered_nodes):
        id_map[node['node_id']] = idx

    edge_list = []
    edge_labels = []

    for edge in edges:
        # 2. Only add the edge if BOTH nodes survived the filtering
        if edge['from'] in id_map and edge['to'] in id_map:
            src = id_map[edge['from']]
            dst = id_map[edge['to']]

            edge_list.append([src, dst])
            edge_labels.append(edge["label"])

    # 3. Handle the edge case where no valid edges remain
    if not edge_list:
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float)

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_label = torch.tensor(edge_labels, dtype=torch.float)

    return edge_index, edge_label

In [5]:
def build_knn_edges(features, k=3):
    """
    features: numpy array [num_nodes, feature_dim]
    prvé 2 dimenzie musia byť x_center, y_center
    """
    num_nodes = features.shape[0]

    if num_nodes <= 1:
        return torch.empty((2, 0), dtype=torch.long)

    actual_k = min(k, num_nodes - 1)

    centers = features[:, :2]  # x_center, y_center

    edge_list = []

    for i in range(num_nodes):

        dx = centers[:, 0] - centers[i, 0]
        dy = centers[:, 1] - centers[i, 1]
        
        # anisotropic distance (vertical more important)
        distances = np.sqrt(dx**2 + (1.5 * dy)**2)

        nearest = np.argsort(distances)[1:actual_k+1]  # jump yourself

        for j in nearest:
            edge_list.append([i, j])
            edge_list.append([j, i])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index

# Load all data

In [22]:
def load_all_data(json_folder):
    all_graphs = []

    for filename in os.listdir(json_folder):
        if filename.endswith(".json"):
            with open(os.path.join(json_folder, filename)) as f:
                graph = json.load(f)

            # features + filtered nodes
            features, filtered_nodes = build_node_features(graph)
            x = torch.tensor(features, dtype=torch.float)

            # propagation edges
            knn_edges = build_knn_edges(features, k=8)

            # edge targets
            true_edge_index, true_edge_labels = build_edge_index(graph['edges'], filtered_nodes)

            # node labels based on filtered nodes
            node_labels = torch.tensor(
                [1 if n['category_id'] == 9 else 0 for n in filtered_nodes],
                dtype=torch.long
            )

            y_nodes=node_labels,
            data = Data(
                x=x,
                edge_index=knn_edges,
                edge_index_targets=true_edge_index,
                y_edges=true_edge_labels
            )

            all_graphs.append(data)

    return all_graphs

In [28]:
import os
# Load graphs
train_graphs = load_all_data("data/DocLayNet/train_data")
val_graphs = load_all_data("data/DocLayNet/val_data")
test_graphs = load_all_data("data/DocLayNet/test_data")

# Improve dataset

In [2]:
import json

with open("data/DocLayNet/COCO/train.json") as f:
    train_coco = json.load(f)

images = train_coco['images']
print(len(images))

with open("data/DocLayNet/COCO/val.json") as f:
    val_coco = json.load(f)

images = val_coco['images']
print(len(images))

with open("data/DocLayNet/COCO/test.json") as f:
    test_coco = json.load(f)

images = test_coco['images']
print(len(images))

69375
6489
4999


In [30]:
train_len = len(train_graphs)
val_len = len(val_graphs)
test_len = len(test_graphs)

overall = train_len + val_len + test_len

print(f"Size of train data: {train_len}, {train_len/overall * 100:.2f}% of dataset")
print(f"Size of val data: {val_len}, {val_len/overall * 100:.2f}% of dataset")
print(f"Size of test data: {test_len}, {test_len/overall * 100:.2f}% of dataset")

Size of train data: 69103, 85.76% of dataset
Size of val data: 6480, 8.04% of dataset
Size of test data: 4994, 6.20% of dataset


In [31]:
TRAIN_PATH = "data/DocLayNet/train_data"
VAL_PATH = "data/DocLayNet/val_data"
TEST_PATH = "data/DocLayNet/test_data"

pages_to_remove_train = []
pages_to_remove_val = []
pages_to_remove_test = []
pages_to_reduce_train = []

for folder in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    pages_to_remove = []
    
    table = 0
    figure = 0
    formula = 0
    other = 0
    caption = 0

    table_caption = 0
    figure_caption = 0
    formula_caption = 0
    other_caption = 0 

    total_files = 0
    empty_nodes_pages = 0   
    empty_edges_pages = 0 
    lone_caption_pages = 0

    for filename in os.listdir(folder):
        if filename.endswith(".json"):
            with open(os.path.join(folder, filename), encoding='utf-8') as f:
                graph = json.load(f)
                
                nodes = graph.get("nodes", [])
                edges = graph.get("edges", [])

                if not nodes:
                    empty_nodes_pages += 1
                    if folder == TRAIN_PATH:
                        pages_to_remove_train.append(filename)
                    elif folder == VAL_PATH:
                        pages_to_remove_val.append(filename)
                    elif folder == TEST_PATH:
                        pages_to_remove_test.append(filename)
                    continue
                    
                for node in nodes:
                    cat = node['category_id']
                    if cat == 1: caption += 1
                    elif cat == 3: formula += 1
                    elif cat == 7: figure += 1
                    elif cat == 9: table += 1
                    else: other += 1
                
                has_connection = any(edge.get('label') == 1 for edge in edges)
                if not has_connection:
                    empty_edges_pages += 1
                    if folder == TRAIN_PATH:
                        pages_to_reduce_train.append(filename)
                    

                targets = [n for n in nodes if n['category_id'] in [3, 7, 9]]
                has_caption = any(n['category_id'] == 1 for n in nodes)
                if has_caption and not targets:
                    if folder == TRAIN_PATH:
                        pages_to_remove_train.append(filename)
                    elif folder == VAL_PATH:
                        pages_to_remove_val.append(filename)
                    elif folder == TEST_PATH:
                        pages_to_remove_test.append(filename)
                    lone_caption_pages += 1
                
                nodes_map = {node['node_id']: node for node in graph['nodes']}
                for edge in edges:
                    if edge['label'] == 1:
                        node_id = edge['to']
                
                        if nodes_map[node_id]['category_id'] == 3:
                           formula_caption += 1
                        elif nodes_map[node_id]['category_id'] == 7:
                           figure_caption += 1
                        elif nodes_map[node_id]['category_id'] == 9:
                           table_caption += 1
                        else:
                            other_caption += 1
                
    # Calculate total connections first for clarity
    total_connections = formula_caption + figure_caption + table_caption + other_caption
    
    print(f"""
    {'='*50}
    DATA ANALYSIS SUMMARY FOR {folder}
    {'='*50}
    Total Files (Pages):     {total_files}
    Empty Pages (No Nodes):  {empty_nodes_pages}
    Pages with No Links:     {empty_edges_pages}
    Pages with Lone Caption: {lone_caption_pages}
    --------------------------------------------------
    Total Captions: {caption:<5} | Total Connections: {total_connections}
    --------------------------------------------------
    Formulas      | {formula:<10} | {formula_caption}
    Figures       | {figure:<10} | {figure_caption}
    Tables        | {table:<10} | {table_caption}
    Others        | {other:<10} | {other_caption}
    {'='*50}
    """)
            


    DATA ANALYSIS SUMMARY FOR data/DocLayNet/train_data
    Total Files (Pages):     0
    Empty Pages (No Nodes):  0
    Pages with No Links:     57290
    Pages with Lone Caption: 451
    --------------------------------------------------
    Total Captions: 19218 | Total Connections: 18305
    --------------------------------------------------
    Formulas      | 21167      | 110
    Figures       | 39667      | 15493
    Tables        | 30070      | 2702
    Others        | 831001     | 0
    

    DATA ANALYSIS SUMMARY FOR data/DocLayNet/val_data
    Total Files (Pages):     0
    Empty Pages (No Nodes):  0
    Pages with No Links:     5429
    Pages with Lone Caption: 16
    --------------------------------------------------
    Total Captions: 1763  | Total Connections: 1735
    --------------------------------------------------
    Formulas      | 1894       | 6
    Figures       | 2775       | 1450
    Tables        | 2269       | 279
    Others        | 91115      | 0
    



The dataset contains some discrepencies where caption is lonely but in reality there is element it belongs to, however the elemnt is not labeled. Other problem is text like citation or block of text with particular caption, we ignore these problems as 97% of caption are connected mostly correctly.

There are countable pages with no edges, specificaly in train data there is 82.5% of pages without any relationship. There is weak edge prediction. The training is biased.

downsample data to 30% no relationship and 70% with caption links
The object distribution of connections for training data is following: 0.6% formulas, 14.7% tables and 84.6% figures. For now we ignore everything or later predict caption-figure and caption-table and not binary as 0 or 1 for relationship.

Also the train/val/test split is not ideal.

increasing k to 8 or more
removing tiny text boxes to reduce noise significantly - we are going to remove if area is lower than 0.001 which corresponds to 3.3% text blocks
remove empty pages
remove lone caption pages
split data to 70:15:15
Lets first analyze the tiny text boxes how many do we have in train dataset for example

# Delete empty pages, pages with lone captions

In [32]:
cleanup_map = {
    TRAIN_PATH: pages_to_remove_train,
    VAL_PATH: pages_to_remove_val,
    TEST_PATH: pages_to_remove_test
}

for folder_path, files_to_delete in cleanup_map.items():
    print(f"Removing {len(files_to_delete)} files in {folder_path}...")
    for filename in files_to_delete:
        file_path = os.path.join(folder_path, filename)
        if os.path.exists(file_path):
            os.remove(file_path)
    print("Done.")

Removing 451 files in data/DocLayNet/train_data...
Done.
Removing 16 files in data/DocLayNet/val_data...
Done.
Removing 31 files in data/DocLayNet/test_data...
Done.


In [34]:
import random

files_to_remove = pages_to_reduce_train
PAGES_TO_KEEP = 12000

files_to_remove = [f for f in files_to_remove if os.path.exists(os.path.join(TRAIN_PATH, f))]

total_empty_pages = len(files_to_remove)
print(f"Number of empty pages (existing): {total_empty_pages}")

if total_empty_pages > PAGES_TO_KEEP:
    num_to_delete = total_empty_pages - PAGES_TO_KEEP

    no_fig_no_table = []
    has_fig_or_table = []

    for filename in files_to_remove:
        file_path = os.path.join(TRAIN_PATH, filename)
        with open(file_path, encoding='utf-8') as f:
            graph = json.load(f)
        nodes = graph.get("nodes", [])
        cats = {node['category_id'] for node in nodes}
        if 7 not in cats and 9 not in cats:
            no_fig_no_table.append(filename)
        else:
            has_fig_or_table.append(filename)

    print(f"Pages without figures & tables: {len(no_fig_no_table)}")
    print(f"Pages with at least figure or table: {len(has_fig_or_table)}")

    actually_delete = no_fig_no_table[:num_to_delete]
    if len(actually_delete) < num_to_delete:
        still_needed = num_to_delete - len(actually_delete)
        actually_delete += random.sample(has_fig_or_table, still_needed)

    print(f"We keep: {PAGES_TO_KEEP}")
    print(f"We delete: {len(actually_delete)} files from disk...")

    deleted_count = 0
    for filename in actually_delete:
        file_path = os.path.join(TRAIN_PATH, filename)
        if os.path.exists(file_path):
            os.remove(file_path)
            deleted_count += 1

    print(f"Successfully deleted {deleted_count} files")
else:
    print("Less no connection pages than the limit")

Number of empty pages (existing): 56839
Pages without figures & tables: 32294
Pages with at least figure or table: 24545
We keep: 12000
We delete: 44839 files from disk...
Successfully deleted 44839 files


In [35]:
TRAIN_PATH = "data/DocLayNet/train_data"
VAL_PATH = "data/DocLayNet/val_data"
TEST_PATH = "data/DocLayNet/test_data"

for folder in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    table = 0
    figure = 0
    other = 0
    caption = 0
    table_caption = 0
    figure_caption = 0
    total_files = 0
    empty_nodes_pages = 0
    empty_edges_pages = 0

    for filename in os.listdir(folder):
        if filename.endswith(".json"):
            total_files += 1
            with open(os.path.join(folder, filename), encoding='utf-8') as f:
                graph = json.load(f)

            nodes = graph.get("nodes", [])
            edges = graph.get("edges", [])

            if not nodes:
                empty_nodes_pages += 1
                continue

            for node in nodes:
                cat = node['category_id']
                if cat == 1: caption += 1
                elif cat == 7: figure += 1
                elif cat == 9: table += 1
                else: other += 1

            has_connection = any(edge.get('label') == 1 for edge in edges)
            if not has_connection:
                empty_edges_pages += 1

            nodes_map = {node['node_id']: node for node in nodes}
            for edge in edges:
                if edge['label'] == 1:
                    target_cat = nodes_map[edge['to']]['category_id']
                    if target_cat == 7: figure_caption += 1
                    elif target_cat == 9: table_caption += 1

    print(f"""
    {'='*50}
    DATA ANALYSIS SUMMARY FOR {folder}
    {'='*50}
    Total Files (Pages):     {total_files}
    Empty Pages (No Nodes):  {empty_nodes_pages}
    Pages with No Links:     {empty_edges_pages}
    --------------------------------------------------
    Total Captions:          {caption}
    --------------------------------------------------
    Figures       | {figure:<10} | figure-caption: {figure_caption}
    Tables        | {table:<10} | table-caption:  {table_caption}
    Others        | {other}
    {'='*50}
    """)


    DATA ANALYSIS SUMMARY FOR data/DocLayNet/train_data
    Total Files (Pages):     23813
    Empty Pages (No Nodes):  0
    Pages with No Links:     12000
    --------------------------------------------------
    Total Captions:          18492
    --------------------------------------------------
    Figures       | 28234      | figure-caption: 15493
    Tables        | 16341      | table-caption:  2702
    Others        | 227727
    

    DATA ANALYSIS SUMMARY FOR data/DocLayNet/val_data
    Total Files (Pages):     6464
    Empty Pages (No Nodes):  0
    Pages with No Links:     5413
    --------------------------------------------------
    Total Captions:          1740
    --------------------------------------------------
    Figures       | 2775       | figure-caption: 1450
    Tables        | 2269       | table-caption:  279
    Others        | 92834
    

    DATA ANALYSIS SUMMARY FOR data/DocLayNet/test_data
    Total Files (Pages):     4963
    Empty Pages (No Nodes):  0

In [7]:
import json
import os
from collections import defaultdict

COCO_PATH = "data/DocLayNet/COCO"
TRAIN_PATH = "data/DocLayNet/train_data"
VAL_PATH   = "data/DocLayNet/val_data"
TEST_PATH  = "data/DocLayNet/test_data"

# Load COCO files
with open(os.path.join(COCO_PATH, "train.json"), encoding='utf-8') as f:
    train_coco = json.load(f)
with open(os.path.join(COCO_PATH, "val.json"), encoding='utf-8') as f:
    val_coco = json.load(f)
with open(os.path.join(COCO_PATH, "test.json"), encoding='utf-8') as f:
    test_coco = json.load(f)

# Group of documents per set
def analyze_set(coco, folder, set_name):
    doc_groups = defaultdict(list)
    for img in coco["images"]:
        doc_groups[img["doc_name"]].append(img["id"])

    result = {}
    for doc_name, img_ids in doc_groups.items():
        total_pages = 0
        linked_pages = 0
        for img_id in img_ids:
            fname = f"graph_{img_id:06d}.json"
            fpath = os.path.join(folder, fname)
            if not os.path.exists(fpath):
                continue
            total_pages += 1
            with open(fpath, encoding='utf-8') as f:
                graph = json.load(f)
            edges = graph.get("edges", [])
            if any(e.get("label") == 1 for e in edges):
                linked_pages += 1
        result[doc_name] = {
            "img_ids": img_ids,
            "total_pages": total_pages,
            "linked_pages": linked_pages,
        }

    total_p = sum(v["total_pages"] for v in result.values())
    linked_p = sum(v["linked_pages"] for v in result.values())
    print(f"\n{set_name}: {len(result)} dokumentov | {total_p} stránok | {linked_p} linked ({100*linked_p/max(total_p,1):.1f}%)")
    return result

train_docs = analyze_set(train_coco, TRAIN_PATH, "TRAIN")
val_docs   = analyze_set(val_coco,   VAL_PATH,   "VAL")
test_docs  = analyze_set(test_coco,  TEST_PATH,  "TEST")

# Top 20 train documents with most linked pages
print("\nTop 20 train documents (most linked pages):")
print(f"{'doc_name':<45} {'pages':>6} {'linked':>7} {'%':>6}")
print("-" * 65)
sorted_docs = sorted(train_docs.items(), key=lambda x: x[1]["linked_pages"], reverse=True)
for doc_name, info in sorted_docs[:20]:
    pct = 100 * info["linked_pages"] / max(info["total_pages"], 1)
    print(f"{doc_name:<45} {info['total_pages']:>6} {info['linked_pages']:>7} {pct:>5.1f}%")


TRAIN: 2587 dokumentov | 19106 stránok | 7383 linked (38.6%)

VAL: 160 dokumentov | 6464 stránok | 1051 linked (16.3%)

TEST: 196 dokumentov | 4963 stránok | 996 linked (20.1%)

Top 20 train documents (most linked pages):
doc_name                                       pages  linked      %
-----------------------------------------------------------------
refman-8.0-en.pdf                               1345     296  22.0%
prh_change1.pdf                                  187     141  75.4%
Pascal - Manual & Report.pdf                      72      49  68.1%
TSX_4503_2014.pdf                                 83      29  34.9%
TSX_BMO_2002.pdf                                  55      27  49.1%
TSX_FTS_2001.pdf                                  34      25  73.5%
NYSE_SSL_2010.pdf                                 46      25  54.3%
0805.0984.pdf                                     27      25  92.6%
1408.5954.pdf                                     26      25  96.2%
airship_pilot_manual.pdf       

In [39]:
TARGET_VAL_RATIO  = 0.40
TARGET_TEST_RATIO = 0.40

cur_val_linked  = val_linked
cur_val_total   = val_total
cur_test_linked = test_linked
cur_test_total  = test_total

docs_to_val  = []
docs_to_test = []

for doc_name, info in candidates:
    lp = info["linked_pages"]
    tp = info["total_pages"]

    val_ratio  = cur_val_linked  / max(cur_val_total,  1)
    test_ratio = cur_test_linked / max(cur_test_total, 1)

    if val_ratio < TARGET_VAL_RATIO:
        docs_to_val.append((doc_name, info))
        cur_val_linked += lp
        cur_val_total  += tp
    elif test_ratio < TARGET_TEST_RATIO:
        docs_to_test.append((doc_name, info))
        cur_test_linked += lp
        cur_test_total  += tp
    else:
        break

train_linked_after = sum(v["linked_pages"] for v in train_docs.values()) - sum(i["linked_pages"] for _, i in docs_to_val + docs_to_test)
train_total_after  = sum(v["total_pages"]  for v in train_docs.values()) - sum(i["total_pages"]  for _, i in docs_to_val + docs_to_test)

print(f"Train after change: {train_total_after} stránok, {train_linked_after} linked ({100*train_linked_after/train_total_after:.1f}%)")
print(f"Val after change: {cur_val_total} stránok, {cur_val_linked} linked ({100*cur_val_linked/cur_val_total:.1f}%)")
print(f"Test after change: {cur_test_total} stránok, {cur_test_linked} linked ({100*cur_test_linked/cur_test_total:.1f}%)")

print(f"\Moving to VAL:  {len(docs_to_val)} documents")
print(f"Moving to TEST: {len(docs_to_test)} documents")

Train po presune: 19106 stránok, 7383 linked (38.6%)
Val   po presune: 9239 stránok, 3706 linked (40.1%)
Test  po presune: 6895 stránok, 2771 linked (40.2%)

Presúvame do VAL:  14 dokumentov
Presúvame do TEST: 31 dokumentov


In [40]:
import shutil

def move_docs_to_set(docs_to_move, src_folder, dst_folder, coco):
    doc_to_ids = defaultdict(list)
    for img in coco["images"]:
        doc_to_ids[img["doc_name"]].append(img["id"])

    moved = 0
    not_found = 0
    for doc_name, info in docs_to_move:
        img_ids = doc_to_ids[doc_name]
        for img_id in img_ids:
            fname = f"graph_{img_id:06d}.json"
            src = os.path.join(src_folder, fname)
            dst = os.path.join(dst_folder, fname)
            if os.path.exists(src):
                shutil.move(src, dst)
                moved += 1
            else:
                not_found += 1

    print(f"  Presunuté: {moved} súborov | Nenájdené: {not_found}")

print("Moving to VAL...")
move_docs_to_set(docs_to_val, TRAIN_PATH, VAL_PATH, train_coco)

print("Moving to TEST...")
move_docs_to_set(docs_to_test, TRAIN_PATH, TEST_PATH, train_coco)

print("\Done! Check up:")
for folder, name in [(TRAIN_PATH, "Train"), (VAL_PATH, "Val"), (TEST_PATH, "Test")]:
    count = len([f for f in os.listdir(folder) if f.endswith(".json")])
    print(f"  {name}: {count} documents")

Presúvam do VAL...
  Presunuté: 2775 súborov | Nenájdené: 1439
Presúvam do TEST...
  Presunuté: 1932 súborov | Nenájdené: 1988

Hotovo! Overenie:
  Train: 19106 súborov
  Val: 9239 súborov
  Test: 6895 súborov


In [41]:
for folder, name in [(TRAIN_PATH, "Train"), (VAL_PATH, "Val"), (TEST_PATH, "Test")]:
    table = 0
    figure = 0
    other = 0
    caption = 0
    figure_caption = 0
    table_caption = 0
    total_files = 0
    empty_nodes_pages = 0
    empty_edges_pages = 0

    for filename in os.listdir(folder):
        if not filename.endswith(".json"):
            continue
        total_files += 1
        with open(os.path.join(folder, filename), encoding='utf-8') as f:
            graph = json.load(f)

        nodes = graph.get("nodes", [])
        edges = graph.get("edges", [])

        if not nodes:
            empty_nodes_pages += 1
            continue

        for node in nodes:
            cat = node['category_id']
            if cat == 1:   caption += 1
            elif cat == 7: figure  += 1
            elif cat == 9: table   += 1
            else:          other   += 1

        has_link = any(e.get("label") == 1 for e in edges)
        if not has_link:
            empty_edges_pages += 1

        nodes_map = {n['node_id']: n for n in nodes}
        for edge in edges:
            if edge['label'] == 1:
                cat = nodes_map[edge['to']]['category_id']
                if cat == 7:   figure_caption += 1
                elif cat == 9: table_caption  += 1

    linked_pages = total_files - empty_nodes_pages - empty_edges_pages
    print(f"""
    {'='*55}
    {name.upper()} — {folder}
    {'='*55}
    Total pages:           {total_files}
    Pages without nodes:       {empty_nodes_pages}
    Pages without links:      {empty_edges_pages}
    Pages with links:       {linked_pages} ({100*linked_pages/max(total_files,1):.1f}%)
    --------------------------------------------------
    Caption overal:         {caption}
    --------------------------------------------------
    Figures       | {figure:<10} | figure-caption: {figure_caption}
    Tables        | {table:<10} | table-caption:  {table_caption}
    Others        | {other}
    {'='*55}
    """)


    TRAIN — data/DocLayNet/train_data
    Total stránok:           19106
    Stránky bez uzlov:       0
    Stránky bez linkov:      11723
    Stránky s linkami:       7383 (38.6%)
    --------------------------------------------------
    Captiony celkom:         11659
    --------------------------------------------------
    Figures       | 21852      | figure-caption: 9328
    Tables        | 15249      | table-caption:  2174
    Others        | 183815
    

    VAL — data/DocLayNet/val_data
    Total stránok:           9239
    Stránky bez uzlov:       0
    Stránky bez linkov:      5533
    Stránky s linkami:       3706 (40.1%)
    --------------------------------------------------
    Captiony celkom:         6090
    --------------------------------------------------
    Figures       | 6917       | figure-caption: 5544
    Tables        | 2825       | table-caption:  497
    Others        | 118532
    

    TEST — data/DocLayNet/test_data
    Total stránok:           6895
   